# 01 — Data Overview & Understanding

Profiles the raw Olist tables end to end: shapes, keys, missing values, outliers,
distributions, geography, and customer repeat behaviour. The monthly order-volume
plot at the centre of this notebook is what pins the evaluation window dates used by
every downstream notebook (feature window → label window → holdout; see
`configs/windows.yaml` once set).

Covers capstone Step 2 (dataset overview + inputs for the data dictionary in
`docs/data_dictionary.md`) and produces `data/processed/profile_summary.json`
for the report.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import BR_STATE_REGION, RAW_TABLES, load_table

sns.set_theme(style="whitegrid")
FIG_DIR = REPO_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

tables = {}
for name in RAW_TABLES:
    try:
        tables[name] = load_table(name)
    except FileNotFoundError:
        # geolocation (~61 MB) is not committed; see data/README.md for download
        print(f"NOTE: raw table '{name}' not present locally - skipping its profile")

print(f"Loaded {len(tables)} of {len(RAW_TABLES)} raw tables")

Loaded 9 of 9 raw tables


## Table shapes, keys, and relationships

The dataset is relational: `customers` (one row per order-customer pairing, with
`customer_unique_id` identifying the actual person) → `orders` → `order_items`
(one row **per unit**, so a 2-unit purchase is 2 rows) → `products` / `sellers`,
plus `order_reviews` and `order_payments` keyed by order, `geolocation` keyed by
zip prefix, and the Portuguese→English `category_translation` lookup.

In [2]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        "table": name,
        "rows": len(df),
        "cols": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
profile = pd.DataFrame(profile_rows).set_index("table")
print(profile.to_string())

                         rows  cols  duplicate_rows  memory_mb
table                                                         
customers               99441     5               0       31.1
geolocation           1000163     5          261831      153.2
order_items            112650     7               0       41.3
order_payments         103886     5               0       18.7
order_reviews           99224     7               0       44.8
orders                  99441     8               0       61.8
products                32951     9               0        7.1
sellers                  3095     4               0        0.7
category_translation       71     2               0        0.0


In [3]:
orders = tables["orders"]
customers = tables["customers"]
items = tables["order_items"]
products = tables["products"]
reviews = tables["order_reviews"]

# Key uniqueness and referential integrity - the joins everything else relies on
print("orders.order_id unique:          ", orders["order_id"].is_unique)
print("customers.customer_id unique:    ", customers["customer_id"].is_unique)
print("distinct customer_unique_id:     ", customers["customer_unique_id"].nunique())
print("products.product_id unique:      ", products["product_id"].is_unique)
print("item order_ids in orders:        ", items["order_id"].isin(orders["order_id"]).all())
print("item product_ids in products:    ", items["product_id"].isin(products["product_id"]).all())
print("order_ids with >1 review:        ", int((reviews.groupby("order_id").size() > 1).sum()))

orders.order_id unique:           True
customers.customer_id unique:     True
distinct customer_unique_id:      96096
products.product_id unique:       True
item order_ids in orders:         True
item product_ids in products:     True
order_ids with >1 review:         547


In [4]:
# Missing values - every column with at least one null, per table
missing_rows = []
for name, df in tables.items():
    nulls = df.isna().sum()
    for col, n in nulls[nulls > 0].items():
        missing_rows.append({
            "table": name,
            "column": col,
            "missing": int(n),
            "pct": round(100 * n / len(df), 2),
        })
missing = pd.DataFrame(missing_rows).sort_values("pct", ascending=False)
print(missing.to_string(index=False) if len(missing) else "No missing values found")

        table                        column  missing   pct
order_reviews          review_comment_title    87656 88.34
order_reviews        review_comment_message    58247 58.70
       orders order_delivered_customer_date     2965  2.98
     products           product_name_lenght      610  1.85
     products         product_category_name      610  1.85
     products    product_description_lenght      610  1.85
     products            product_photos_qty      610  1.85
       orders  order_delivered_carrier_date     1783  1.79
       orders             order_approved_at      160  0.16
     products              product_weight_g        2  0.01
     products             product_length_cm        2  0.01
     products             product_height_cm        2  0.01
     products              product_width_cm        2  0.01


## Order timeline & monthly volume

This plot pins the three-window evaluation scheme. The known Olist quirk to
confirm here: order volume collapses after August 2018 (the tail is dropped from
all evaluation windows) and late 2016 is a tiny pilot period.

In [5]:
orders = tables["orders"].copy()
ts_cols = [c for c in orders.columns if c.endswith("_date") or c.endswith("_timestamp")]
for c in ts_cols:
    orders[c] = pd.to_datetime(orders[c])
tables["orders"] = orders

orders["month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
monthly = orders.groupby("month").size()
delivered_monthly = orders[orders["order_status"] == "delivered"].groupby("month").size()

windows_path = REPO_ROOT / "configs" / "windows.yaml"
windows = yaml.safe_load(windows_path.read_text()) if windows_path.exists() else None

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(monthly))
ax.plot(x, monthly.values, marker="o", label="all orders")
ax.plot(x, delivered_monthly.reindex(monthly.index, fill_value=0).values,
        marker=".", label="delivered")
ax.set_xticks(list(x))
ax.set_xticklabels(monthly.index, rotation=60, ha="right", fontsize=8)
if windows:
    month_pos = {m: i for i, m in enumerate(monthly.index)}
    shade = {"feature_window": "tab:green", "label_window": "tab:orange", "holdout": "tab:red"}
    for key, color in shade.items():
        w = windows[key]
        start = month_pos.get(w["start"][:7])
        end = month_pos.get(w["end"][:7])
        if start is not None and end is not None:
            ax.axvspan(start - 0.4, end + 0.4, alpha=0.15, color=color, label=key)
ax.set_title("Olist monthly order volume (evaluation windows shaded)")
ax.set_ylabel("orders")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "01_monthly_order_volume.png", dpi=150)
plt.close(fig)

print(f"first order: {orders['order_purchase_timestamp'].min()}")
print(f"last order:  {orders['order_purchase_timestamp'].max()}")
print(monthly.to_string())

first order: 2016-09-04 21:15:19
last order:  2018-10-17 17:30:18
month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4


In [6]:
status = orders["order_status"].value_counts()
print(status.to_string())
print(f"\ndelivered share: {status.get('delivered', 0) / len(orders):.1%}")

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

delivered share: 97.0%


## Distributions: reviews, price, freight, delivery time

In [7]:
reviews = tables["order_reviews"]
score_dist = reviews["review_score"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=score_dist.index.astype(str), y=score_dist.values, ax=ax, color="tab:blue")
ax.set_title("Review score distribution")
ax.set_xlabel("review_score")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_review_scores.png", dpi=150)
plt.close(fig)
print(score_dist.to_string())
print(f"share >= 4: {(reviews['review_score'] >= 4).mean():.1%}")

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
share >= 4: 77.1%


In [8]:
items = tables["order_items"]
delivered = orders[orders["order_status"] == "delivered"].copy()
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.days

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(items["price"], bins=100, log=True)
axes[0].set_title("Item price (log count)")
axes[0].set_xlabel("BRL")
axes[1].hist(items["freight_value"], bins=100, log=True)
axes[1].set_title("Freight value (log count)")
axes[1].set_xlabel("BRL")
axes[2].hist(delivered["delivery_days"].dropna(), bins=60)
axes[2].set_title("Delivery time (days)")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_price_freight_delivery.png", dpi=150)
plt.close(fig)

# Outlier scan: IQR fence + p99, per the preprocessing plan (no silent handling here -
# this documents scale; winsorisation decisions happen in notebook 02)
for label, series in [
    ("price", items["price"]),
    ("freight_value", items["freight_value"]),
    ("delivery_days", delivered["delivery_days"].dropna()),
]:
    q1, q3 = series.quantile([0.25, 0.75])
    hi = q3 + 1.5 * (q3 - q1)
    print(
        f"{label:14s} median={series.median():8.1f}  p99={series.quantile(0.99):8.1f}  "
        f"max={series.max():9.1f}  >IQR fence: {(series > hi).sum():6d} "
        f"({(series > hi).mean():.1%})"
    )

price          median=    75.0  p99=   890.0  max=   6735.0  >IQR fence:   8427 (7.5%)
freight_value  median=    16.3  p99=    84.5  max=    409.7  >IQR fence:  11613 (10.3%)
delivery_days  median=    10.0  p99=    46.0  max=    209.0  >IQR fence:   5022 (5.2%)


In [9]:
cat_translation = tables["category_translation"]
prod_cat = tables["products"][["product_id", "product_category_name"]].merge(
    cat_translation, on="product_category_name", how="left"
)
item_cats = items.merge(prod_cat, on="product_id", how="left")
untranslated = item_cats.loc[
    item_cats["product_category_name"].notna()
    & item_cats["product_category_name_english"].isna(),
    "product_category_name",
].nunique()
print(f"categories without an English translation: {untranslated}")

top_cats = item_cats["product_category_name_english"].value_counts().head(20)
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, ax=ax, color="tab:blue")
ax.set_title("Top 20 product categories by items sold")
ax.set_xlabel("items")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_top_categories.png", dpi=150)
plt.close(fig)
print(f"total categories: {item_cats['product_category_name_english'].nunique()}")
print(top_cats.head(10).to_string())

categories without an English translation: 2


total categories: 71


product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235


In [10]:
cust_geo = tables["customers"].copy()
cust_geo["region"] = cust_geo["customer_state"].map(BR_STATE_REGION)
state_counts = cust_geo["customer_state"].value_counts()
region_counts = cust_geo["region"].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=state_counts.index, y=state_counts.values, ax=ax, color="tab:blue")
ax.set_title("Customers by state")
ax.set_ylabel("customers")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_customers_by_state.png", dpi=150)
plt.close(fig)

print("Region rollup (fairness-audit grouping):")
print((region_counts / len(cust_geo)).map("{:.1%}".format).to_string())

Region rollup (fairness-audit grouping):

region
Southeast      68.6%
South          14.2%
Northeast       9.4%
Center-West     5.8%
North           1.9%


## Customer repeat behaviour

The single number the whole architecture is designed around: what share of
customers ever come back? Also quantifies the order-items unit-row duplication
that interaction matrices must dedupe.

In [11]:
orders_per_cust = (
    orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)
one_shot_share = (orders_per_cust == 1).mean()
repeat_pool = int((orders_per_cust >= 2).sum())
print(f"customers (unique):        {len(orders_per_cust)}")
print(f"single-order share:        {one_shot_share:.2%}")
print(f"repeat buyers (>=2 orders): {repeat_pool}")
print(f"max orders by one customer: {orders_per_cust.max()}")

pair_rows = len(items)
unique_pairs = items.merge(
    orders.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id')[
        ['order_id', 'customer_unique_id']
    ],
    on='order_id',
)[['customer_unique_id', 'product_id']].drop_duplicates()
print(f"\norder_item rows:            {pair_rows}")
print(f"unique (customer, product): {len(unique_pairs)}")

customers (unique):        96096
single-order share:        96.88%
repeat buyers (>=2 orders): 2997
max orders by one customer: 17



order_item rows:            112650
unique (customer, product): 101987


In [12]:
summary = {
    "first_order": str(orders["order_purchase_timestamp"].min()),
    "last_order": str(orders["order_purchase_timestamp"].max()),
    "monthly_orders": monthly.to_dict(),
    "n_orders": int(len(orders)),
    "delivered_share": float(status.get("delivered", 0) / len(orders)),
    "n_unique_customers": int(len(orders_per_cust)),
    "single_order_share": float(one_shot_share),
    "repeat_buyers": repeat_pool,
    "n_products": int(len(tables["products"])),
    "n_sellers": int(len(tables["sellers"])),
    "n_categories": int(item_cats["product_category_name_english"].nunique()),
    "review_score_share_ge4": float((reviews["review_score"] >= 4).mean()),
    "region_share": (region_counts / len(cust_geo)).round(4).to_dict(),
    "order_item_rows": int(pair_rows),
    "unique_customer_product_pairs": int(len(unique_pairs)),
}
out = PROCESSED_DIR / "profile_summary.json"
out.write_text(json.dumps(summary, indent=2))
print(f"wrote {out}")

wrote C:\Users\jmonz\Documents\GitHub\ecommerce-recommender-capstone\data\processed\profile_summary.json


## Takeaways

- **99,441 orders** spanning Sep 2016 - Oct 2018, but the usable range is
  **2017-01 through 2018-08**: late 2016 is a tiny pilot (329 orders, with a gap in
  Nov 2016) and volume collapses after August 2018 (Sep: 16, Oct: 4). The
  three-window scheme in `configs/windows.yaml` is pinned accordingly.
- **96.9% of the 96,096 customers have exactly one order**; the 2,997 repeat buyers
  are the leave-last-order-out evaluation pool for candidate generation.
- `order_items` has one row **per unit**: 112,650 rows collapse to 101,987 unique
  (customer, product) pairs - interaction matrices must dedupe first.
- Reviews skew positive (77.1% of scores are >= 4) and 97.0% of orders are delivered -
  the `delivered` filter retains almost all data.
- Geography is heavily concentrated: Southeast 68.7% of customers vs North 1.9% -
  the fairness audit's minimum-group-size and region-merge rules will be exercised.
- Missingness is modest and interpretable: review comment fields (most customers
  don't write text), delivery timestamps on undelivered orders, and a small set of
  products with no category/attributes.